# Model 13: Nucleus-Only Decipherment

## Hypothesis
Not all glyphs encode phonetic content. Onset and coda glyphs are **structural markers**
(selected by position but carrying no phonetic value), while **nucleus glyphs** are the
actual content carriers that map to Hebrew characters.

This resolves the word-length paradox:
- A 10-glyph word with ~4 nucleus glyphs → 4 Hebrew characters (reasonable)
- Effective |Σ|/|A| drops from 6.1 to ~2.0 (solvable range)

Consistent with: SilPart (onset/coda structure is real), position ≠ meaning,
73% position-locked glyphs, U-shaped entropy curve.

In [1]:
# ── Cell 1: Imports ──────────────────────────────────────────────────
import re, math, random, json, warnings, time
from collections import Counter, defaultdict
from itertools import combinations
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import requests
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: cuda


In [2]:
# ── Cell 2: Parse Voynich + TP-P20 segmentation + positional classification ──

def parse_sta(filepath):
    tokens = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            if re.match(r'^<[^>]+>\s*<!', line):
                continue
            match = re.match(r'^<[^>]+>\s+(.*)', line)
            if not match:
                continue
            text = match.group(1).strip()
            text = re.sub(r'\{[^}]*\}', '', text)
            text = re.sub(r'<[^>]*>', '', text)
            text = re.sub(r'-', '.', text)
            for w in text.split('.'):
                w = w.strip()
                if not w:
                    continue
                glyphs = re.findall(r'[A-Z][0-9a-z]', w)
                if glyphs:
                    tokens.extend(glyphs)
                    tokens.append('.')
    return tokens


def tokens_to_words(tokens):
    words, current = [], []
    for t in tokens:
        if t == '.':
            if current:
                words.append(current)
                current = []
        else:
            current.append(t)
    if current:
        words.append(current)
    return words


def build_transition_matrix(words):
    all_glyphs = sorted({g for w in words for g in w})
    index = {g: i for i, g in enumerate(all_glyphs)}
    n = len(all_glyphs)
    M = np.zeros((n, n))
    for word in words:
        for i in range(len(word) - 1):
            M[index[word[i]], index[word[i+1]]] += 1
    row_sums = M.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    return M / row_sums, all_glyphs, index


def segment_tp(words, M_prob, index, percentile=20):
    all_tp = []
    for word in words:
        for i in range(len(word) - 1):
            g1, g2 = word[i], word[i+1]
            if g1 in index and g2 in index:
                all_tp.append(M_prob[index[g1], index[g2]])
    threshold = np.percentile(all_tp, percentile)
    result = []
    for word in words:
        if not word:
            continue
        syls = []
        cur = [word[0]]
        for i in range(len(word) - 1):
            p = M_prob[index[word[i]], index[word[i+1]]] if (word[i] in index and word[i+1] in index) else 0.0
            if p < threshold:
                syls.append(cur)
                cur = [word[i+1]]
            else:
                cur.append(word[i+1])
        syls.append(cur)
        result.append(syls)
    return result, threshold


def classify_positions(words_as_syls):
    """Classify each glyph occurrence as onset/nucleus/coda based on syllable position."""
    glyph_pos_counts = defaultdict(lambda: Counter())  # glyph -> {onset:n, nucleus:n, coda:n}
    for word_syls in words_as_syls:
        for syl in word_syls:
            n = len(syl)
            if n == 1:
                glyph_pos_counts[syl[0]]['nucleus'] += 1
            elif n == 2:
                glyph_pos_counts[syl[0]]['onset'] += 1
                glyph_pos_counts[syl[1]]['coda'] += 1
            else:  # 3+
                glyph_pos_counts[syl[0]]['onset'] += 1
                for g in syl[1:-1]:
                    glyph_pos_counts[g]['nucleus'] += 1
                glyph_pos_counts[syl[-1]]['coda'] += 1
    return glyph_pos_counts


def assign_dominant_position(glyph_pos_counts, min_freq=5):
    """For each glyph, determine dominant position and nucleus fraction."""
    glyph_info = {}
    for g, counts in glyph_pos_counts.items():
        total = sum(counts.values())
        if total < min_freq:
            glyph_info[g] = {'dominant': '?', 'pct_nucleus': 0.0, 'total': total}
            continue
        pct = {pos: counts[pos] / total for pos in ['onset', 'nucleus', 'coda']}
        dominant = max(pct, key=pct.get)
        glyph_info[g] = {
            'dominant': dominant,
            'pct_nucleus': pct['nucleus'],
            'pct_onset': pct['onset'],
            'pct_coda': pct['coda'],
            'total': total,
        }
    return glyph_info


# ── Load and segment ──
tokens = parse_sta('corpus/voynich_sta.txt')
words = tokens_to_words(tokens)
VOCAB = sorted({g for w in words for g in w})
print(f'Voynich: {len(words):,} words, {sum(len(w) for w in words):,} glyphs, {len(VOCAB)} types')

M_prob, all_g, idx_g = build_transition_matrix(words)
words_syl, thresh = segment_tp(words, M_prob, idx_g, 20)

# ── Classify positions ──
glyph_pos_counts = classify_positions(words_syl)
glyph_info = assign_dominant_position(glyph_pos_counts)

# ── Separate nucleus vs structural glyphs ──
NUCLEUS_GLYPHS = sorted([g for g, info in glyph_info.items()
                         if info['dominant'] == 'nucleus' and info['total'] >= 5])
ONSET_GLYPHS = sorted([g for g, info in glyph_info.items()
                       if info['dominant'] == 'onset' and info['total'] >= 5])
CODA_GLYPHS = sorted([g for g, info in glyph_info.items()
                      if info['dominant'] == 'coda' and info['total'] >= 5])
STRUCTURAL_GLYPHS = set(ONSET_GLYPHS + CODA_GLYPHS)

print(f'\nPositional classification:')
print(f'  Onset:   {len(ONSET_GLYPHS)} types')
print(f'  Nucleus: {len(NUCLEUS_GLYPHS)} types')
print(f'  Coda:    {len(CODA_GLYPHS)} types')
print(f'  Total structural (onset+coda): {len(STRUCTURAL_GLYPHS)}')

# ── Extract nucleus-only content from each word ──
def extract_nucleus_glyphs(word_syls):
    """Extract only nucleus-position glyphs from a syllabified word."""
    nucleus = []
    for syl in word_syls:
        n = len(syl)
        if n == 1:
            nucleus.append(syl[0])  # single glyph = nucleus
        elif n == 2:
            pass  # onset + coda, no nucleus
        else:  # 3+
            nucleus.extend(syl[1:-1])  # middle glyphs = nucleus
    return nucleus


words_nucleus = [extract_nucleus_glyphs(ws) for ws in words_syl]
# Some words may have 0 nucleus glyphs (all 2-glyph syllables)
# For these, fall back to the full word
words_content = []
n_fallback = 0
for i, nuc in enumerate(words_nucleus):
    if len(nuc) > 0:
        words_content.append(nuc)
    else:
        # Fallback: use all glyphs (rare case)
        words_content.append(words[i])
        n_fallback += 1

content_lengths = [len(w) for w in words_content]
orig_lengths = [len(w) for w in words]
CONTENT_VOCAB = sorted({g for w in words_content for g in w})

print(f'\n=== Nucleus extraction ===')
print(f'Content glyphs vocabulary: {len(CONTENT_VOCAB)} types')
print(f'Mean word length: {np.mean(orig_lengths):.2f} → {np.mean(content_lengths):.2f} (nucleus only)')
print(f'Words with 0 nucleus (fallback): {n_fallback} ({100*n_fallback/len(words):.1f}%)')
print(f'\nLength distribution comparison:')
for l in range(1, 11):
    n_orig = sum(1 for x in orig_lengths if x == l)
    n_nuc = sum(1 for x in content_lengths if x == l)
    print(f'  {l:2d}: {n_orig:6d} → {n_nuc:6d}')

Voynich: 37,087 words, 157,254 glyphs, 166 types

Positional classification:
  Onset:   42 types
  Nucleus: 28 types
  Coda:    5 types
  Total structural (onset+coda): 47

=== Nucleus extraction ===
Content glyphs vocabulary: 120 types
Mean word length: 4.24 → 2.22 (nucleus only)
Words with 0 nucleus (fallback): 4394 (11.8%)

Length distribution comparison:
   1:    883 →  11266
   2:   3529 →  13380
   3:   8143 →   7168
   4:   9328 →   4144
   5:   8269 →    690
   6:   4342 →    321
   7:   1605 →     64
   8:    516 →     36
   9:    222 →     12
  10:    113 →      4


In [3]:
# ── Cell 3: Download Hebrew + train LSTM ─────────────────────────────

def download_hebrew():
    books = ['Genesis', 'Exodus', 'Leviticus', 'Numbers', 'Deuteronomy']
    base_url = ('https://raw.githubusercontent.com/Sefaria/Sefaria-Export/'
                'master/json/Tanakh/Torah/{}/Hebrew/Tanach%20with%20Text%20Only.json')
    all_words = []
    for book in books:
        url = base_url.format(book)
        print(f'  Downloading {book}...')
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        for chapter in data['text']:
            for verse in chapter:
                text = re.sub(r'[\u0591-\u05C7]', '', verse)
                ws = re.findall(r'[\u05D0-\u05EA]+', text)
                for w in ws:
                    chars = list(w)
                    if 2 <= len(chars) <= 15:
                        all_words.append(chars)
    print(f'  Hebrew total: {len(all_words):,} words')
    return all_words

print('Downloading Hebrew Bible (Torah)...')
hebrew_words_full = download_hebrew()

# Subsample
rng_sub = np.random.default_rng(42)
if len(hebrew_words_full) > len(words):
    idx_sub = rng_sub.choice(len(hebrew_words_full), size=len(words), replace=False)
    hebrew_words = [hebrew_words_full[i] for i in sorted(idx_sub)]
else:
    hebrew_words = hebrew_words_full
target_chars = sorted({c for w in hebrew_words for c in w})
print(f'Hebrew subsample: {len(hebrew_words):,} words, {len(target_chars)} chars')
print(f'Mean Hebrew word length: {np.mean([len(w) for w in hebrew_words]):.2f}')
print(f'Mean Voynich content length: {np.mean(content_lengths):.2f}')
print(f'|Σ_content|/|A|: {len(CONTENT_VOCAB)}/{len(target_chars)} = {len(CONTENT_VOCAB)/len(target_chars):.2f}')


# ── LSTM ──
class CharDataset(Dataset):
    def __init__(self, words, char2idx):
        self.samples = []
        bos, eos = char2idx['<w>'], char2idx['</w>']
        for word in words:
            seq = [bos, bos] + [char2idx.get(c, char2idx.get('<unk>', 0)) for c in word] + [eos]
            for i in range(2, len(seq)):
                self.samples.append((seq[i-2:i], seq[i]))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        ctx, tgt = self.samples[idx]
        return torch.tensor(ctx, dtype=torch.long), torch.tensor(tgt, dtype=torch.long)


class NeuralCharLM(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_dim=192, n_layers=2, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=n_layers,
                           dropout=dropout if n_layers > 1 else 0.0, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.char2idx = None
        self.idx2char = None

    def forward(self, x, hidden=None):
        emb = self.dropout(self.embedding(x))
        out, hidden = self.lstm(emb, hidden)
        return self.fc(self.dropout(out)), hidden

    def batch_log_prob(self, words, temperature=1.0):
        self.eval()
        bos, eos = self.char2idx['<w>'], self.char2idx['</w>']
        unk = self.char2idx.get('<unk>', 0)
        with torch.no_grad():
            all_ctx, all_tgt, boundaries = [], [], [0]
            for word in words:
                seq = [bos, bos] + [self.char2idx.get(c, unk) for c in word] + [eos]
                for i in range(2, len(seq)):
                    all_ctx.append([seq[i-2], seq[i-1]])
                    all_tgt.append(seq[i])
                boundaries.append(len(all_ctx))
            if not all_ctx:
                return np.zeros(len(words))
            ctx_t = torch.tensor(all_ctx, device=DEVICE)
            tgt_t = torch.tensor(all_tgt, device=DEVICE)
            all_lps = []
            for s in range(0, len(all_ctx), 16384):
                e = min(s + 16384, len(all_ctx))
                logits, _ = self.forward(ctx_t[s:e])
                lp = torch.log_softmax(logits[:, -1, :] / temperature, dim=-1)
                all_lps.append(lp.gather(1, tgt_t[s:e].unsqueeze(1)).squeeze(1))
            all_lps = torch.cat(all_lps).cpu().numpy() / math.log(2)
            word_lps = np.zeros(len(words))
            for wi in range(len(words)):
                s, e = boundaries[wi], boundaries[wi+1]
                if s < e: word_lps[wi] = all_lps[s:e].sum()
            return word_lps


# Build vocab and train
special = ['<pad>', '<w>', '</w>', '<unk>']
char_vocab = special + target_chars
char2idx = {c: i for i, c in enumerate(char_vocab)}
idx2char = {i: c for c, i in char2idx.items()}

rng_split = np.random.default_rng(42)
n_val = max(1000, len(hebrew_words) // 10)
idx_all = rng_split.permutation(len(hebrew_words))
train_words = [hebrew_words[i] for i in idx_all[n_val:]]
val_words = [hebrew_words[i] for i in idx_all[:n_val]]

train_dl = DataLoader(CharDataset(train_words, char2idx), batch_size=512, shuffle=True)
val_dl = DataLoader(CharDataset(val_words, char2idx), batch_size=1024)

neural_lm = NeuralCharLM(len(char_vocab)).to(DEVICE)
neural_lm.char2idx = char2idx
neural_lm.idx2char = idx2char
optimizer = torch.optim.Adam(neural_lm.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

print(f'\nTraining LSTM...')
best_val_loss = float('inf')
patience, best_state = 0, None
for epoch in range(30):
    neural_lm.train()
    tl, nb = 0, 0
    for ctx_b, tgt_b in train_dl:
        ctx_b, tgt_b = ctx_b.to(DEVICE), tgt_b.to(DEVICE)
        logits, _ = neural_lm(ctx_b)
        loss = criterion(logits[:, -1, :], tgt_b)
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(neural_lm.parameters(), 1.0)
        optimizer.step()
        tl += loss.item(); nb += 1
    neural_lm.eval()
    vl, vn = 0, 0
    with torch.no_grad():
        for ctx_b, tgt_b in val_dl:
            ctx_b, tgt_b = ctx_b.to(DEVICE), tgt_b.to(DEVICE)
            logits, _ = neural_lm(ctx_b)
            vl += criterion(logits[:, -1, :], tgt_b).item(); vn += 1
    avg_val = vl / max(vn, 1)
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        best_state = {k: v.cpu().clone() for k, v in neural_lm.state_dict().items()}
        patience = 0
    else:
        patience += 1
    if epoch % 5 == 0 or patience > 0:
        print(f'  Epoch {epoch:2d}: train={tl/nb:.4f}, val={avg_val:.4f}, patience={patience}')
    if patience >= 5:
        print(f'  Early stopping at epoch {epoch}')
        break
neural_lm.load_state_dict(best_state)
neural_lm = neural_lm.to(DEVICE)
print(f'Best val loss: {best_val_loss:.4f}')

  Hebrew total: 80,051 words
Hebrew subsample: 37,087 words, 27 chars
Mean Hebrew word length: 3.81
Mean Voynich content length: 2.22
|Σ_content|/|A|: 120/27 = 4.44

Training LSTM...
  Epoch  0: train=2.5242, val=2.2756, patience=0
  Epoch  5: train=2.0836, val=2.0451, patience=0
  Epoch 10: train=2.0482, val=2.0233, patience=0
  Epoch 14: train=2.0354, val=2.0188, patience=1
  Epoch 15: train=2.0344, val=2.0163, patience=0
  Epoch 16: train=2.0321, val=2.0165, patience=1
  Epoch 18: train=2.0285, val=2.0156, patience=1
  Epoch 20: train=2.0256, val=2.0126, patience=1
  Epoch 24: train=2.0211, val=2.0116, patience=1
  Epoch 25: train=2.0204, val=2.0097, patience=0
  Epoch 27: train=2.0204, val=2.0101, patience=1
  Epoch 28: train=2.0193, val=2.0101, patience=2
  Epoch 29: train=2.0178, val=2.0104, patience=3
Best val loss: 2.0088


In [4]:
# ── Cell 4: Nucleus-only mapping state + scoring ─────────────────────

class NucleusMappingState:
    """Maps only nucleus-position glyphs to Hebrew characters.
    Onset/coda glyphs are treated as NULL (structural markers)."""
    
    def __init__(self, content_glyphs, target_chars):
        self.content_glyphs = list(content_glyphs)
        self.target_chars = list(target_chars)
        self.mapping = {}  # content glyph → Hebrew char
    
    def set_mapping(self, mapping_dict):
        self.mapping = dict(mapping_dict)
    
    def propose_change(self, rng):
        new = NucleusMappingState(self.content_glyphs, self.target_chars)
        new.mapping = dict(self.mapping)
        idx = rng.integers(len(self.content_glyphs))
        glyph = self.content_glyphs[idx]
        new.mapping[glyph] = self.target_chars[rng.integers(len(self.target_chars))]
        return new, glyph
    
    def decode_word(self, content_word):
        """Decode a nucleus-only word (list of content glyphs) → Hebrew chars."""
        return [self.mapping.get(g, '?') for g in content_word]
    
    def decode_corpus(self, content_words):
        return [self.decode_word(w) for w in content_words]


def init_frequency_mapping(content_words, content_glyphs, hebrew_words, target_chars):
    """Match frequency ranks."""
    v_freq = Counter(g for w in content_words for g in w)
    h_freq = Counter(c for w in hebrew_words for c in w)
    v_ranked = sorted(content_glyphs, key=lambda g: v_freq.get(g, 0), reverse=True)
    h_ranked = sorted(target_chars, key=lambda c: h_freq.get(c, 0), reverse=True)
    return {v_ranked[i]: h_ranked[i % len(h_ranked)] for i in range(len(v_ranked))}


def build_glyph_word_index(content_words):
    idx = defaultdict(list)
    for wi, word in enumerate(content_words):
        for g in set(word):
            idx[g].append(wi)
    return idx


def score_lm(state, content_sample, neural_lm, temperature):
    decoded = state.decode_corpus(content_sample)
    word_lps = neural_lm.batch_log_prob(decoded, temperature)
    return float(np.mean(word_lps)), word_lps


def score_entropy(state):
    counts = Counter(state.mapping.values())
    if len(counts) <= 1: return 0.0
    total = sum(counts.values())
    probs = np.array([c/total for c in counts.values()])
    probs = probs[probs > 0]
    H = -np.sum(probs * np.log2(probs))
    return H / np.log2(len(counts))


def score_total(state, content_sample, neural_lm, temperature,
                w_lm=2.0, w_ent=0.3):
    s_lm, word_lps = score_lm(state, content_sample, neural_lm, temperature)
    s_ent = score_entropy(state)
    return w_lm * s_lm + w_ent * s_ent, word_lps


def score_incremental(proposal, changed_glyph, content_sample, neural_lm,
                      temperature, glyph_word_idx, prev_word_lps,
                      w_lm=2.0, w_ent=0.3):
    new_lps = prev_word_lps.copy()
    affected = glyph_word_idx.get(changed_glyph, [])
    if affected:
        decoded = [proposal.decode_word(content_sample[wi]) for wi in affected]
        scores = neural_lm.batch_log_prob(decoded, temperature)
        for j, wi in enumerate(affected):
            new_lps[wi] = scores[j]
    s_lm = float(np.mean(new_lps))
    s_ent = score_entropy(proposal)
    return w_lm * s_lm + w_ent * s_ent, new_lps


print(f'Mapping space: {len(CONTENT_VOCAB)} content glyphs → {len(target_chars)} Hebrew chars')
print(f'Ratio |Σ|/|A|: {len(CONTENT_VOCAB)/len(target_chars):.2f}')

Mapping space: 120 content glyphs → 27 Hebrew chars
Ratio |Σ|/|A|: 4.44


In [5]:
# ── Cell 5: Multi-seed EM ────────────────────────────────────────────

N_SEEDS = 10
T_VALUES = [2.0, 3.0]
SAMPLE_EM = 3000
EM_ITERS = 1500
EM_PATIENCE = 80
PROPOSALS_PER_ITER = 200

# Subsample
rng_em = np.random.default_rng(42)
if len(words_content) > SAMPLE_EM:
    idx_em = rng_em.choice(len(words_content), size=SAMPLE_EM, replace=False)
    content_em = [words_content[i] for i in idx_em]
else:
    content_em = words_content

glyph_word_idx = build_glyph_word_index(content_em)
base_mapping = init_frequency_mapping(words_content, CONTENT_VOCAB, hebrew_words, target_chars)

all_em_results = []
for T in T_VALUES:
    print(f'\n=== Temperature T={T} ===')
    for seed_idx in range(N_SEEDS):
        rng = np.random.default_rng(42 + seed_idx * 137 + int(T * 1000))
        
        state = NucleusMappingState(CONTENT_VOCAB, target_chars)
        if seed_idx < 5:
            mapping_init = dict(base_mapping)
            n_scramble = max(1, len(CONTENT_VOCAB) // 3)
            scramble_idx = rng.choice(len(CONTENT_VOCAB), size=n_scramble, replace=False)
            for si in scramble_idx:
                g = CONTENT_VOCAB[si]
                mapping_init[g] = target_chars[rng.integers(len(target_chars))]
            state.set_mapping(mapping_init)
        else:
            state.set_mapping({g: target_chars[rng.integers(len(target_chars))] for g in CONTENT_VOCAB})
        
        best_score, best_lps = score_total(state, content_em, neural_lm, T)
        best_state = state
        no_improve = 0
        
        t0 = time.time()
        for it in range(EM_ITERS):
            improved = False
            for _ in range(PROPOSALS_PER_ITER):
                proposal, changed = best_state.propose_change(rng)
                prop_score, prop_lps = score_incremental(
                    proposal, changed, content_em, neural_lm, T,
                    glyph_word_idx, best_lps)
                if prop_score > best_score:
                    best_state, best_score, best_lps = proposal, prop_score, prop_lps
                    improved = True
            if not improved:
                no_improve += 1
                if no_improve >= EM_PATIENCE: break
            else:
                no_improve = 0
        
        elapsed = time.time() - t0
        K_eff = len(set(best_state.mapping.values()))
        result = {'seed': seed_idx, 'T': T, 'score': best_score,
                  'K_eff': K_eff, 'elapsed': elapsed, 'state': best_state,
                  'word_lps': best_lps, 'iter': it + 1}
        all_em_results.append(result)
        
        init_type = 'freq+scramble' if seed_idx < 5 else 'random'
        print(f'  Seed {seed_idx:2d} [{init_type:13s}]: score={best_score:.4f}, '
              f'K={K_eff:3d}, iter={it+1:4d}, {elapsed:.0f}s')

BEST = max(all_em_results, key=lambda r: r['score'])
print(f'\nBest: T={BEST["T"]}, seed={BEST["seed"]}, score={BEST["score"]:.4f}, K_eff={BEST["K_eff"]}')


=== Temperature T=2.0 ===
  Seed  0 [freq+scramble]: score=-24.3474, K= 27, iter= 203, 40s
  Seed  1 [freq+scramble]: score=-24.6648, K= 27, iter= 233, 45s
  Seed  2 [freq+scramble]: score=-24.3475, K= 27, iter= 236, 46s
  Seed  3 [freq+scramble]: score=-24.8303, K= 26, iter= 269, 52s
  Seed  4 [freq+scramble]: score=-24.7392, K= 27, iter= 166, 32s
  Seed  5 [random       ]: score=-24.8242, K= 27, iter= 181, 36s
  Seed  6 [random       ]: score=-24.8757, K= 27, iter= 264, 50s
  Seed  7 [random       ]: score=-24.8659, K= 27, iter= 226, 42s
  Seed  8 [random       ]: score=-25.0032, K= 27, iter= 216, 42s
  Seed  9 [random       ]: score=-24.7683, K= 27, iter= 193, 38s

=== Temperature T=3.0 ===
  Seed  0 [freq+scramble]: score=-25.4328, K= 27, iter= 246, 47s
  Seed  1 [freq+scramble]: score=-25.3123, K= 27, iter= 235, 44s
  Seed  2 [freq+scramble]: score=-25.2540, K= 26, iter= 222, 43s
  Seed  3 [freq+scramble]: score=-25.2152, K= 27, iter= 212, 40s
  Seed  4 [freq+scramble]: score=-25

In [6]:
# ── Cell 6: MCMC multi-chain ─────────────────────────────────────────

T_mcmc = BEST['T']
N_CHAINS = 4
N_ITER = 15000
BURN_IN = 3000
THIN = 5

print(f'MCMC: {N_CHAINS} chains, {N_ITER} iter, T={T_mcmc}')

mcmc_chains = []
for chain_id in range(N_CHAINS):
    rng = np.random.default_rng(42 + chain_id * 999)
    state = NucleusMappingState(CONTENT_VOCAB, target_chars)
    init_map = dict(BEST['state'].mapping)
    # Perturb 10%
    n_perturb = max(1, len(CONTENT_VOCAB) // 10)
    for pi in rng.choice(len(CONTENT_VOCAB), size=n_perturb, replace=False):
        g = CONTENT_VOCAB[pi]
        init_map[g] = target_chars[rng.integers(len(target_chars))]
    state.set_mapping(init_map)
    
    cur_score, cur_lps = score_total(state, content_em, neural_lm, T_mcmc)
    chain_mappings, chain_scores = [], []
    n_accepted = 0
    temp_scale = 1.0
    
    t0 = time.time()
    for it in range(N_ITER):
        proposal, changed = state.propose_change(rng)
        prop_score, prop_lps = score_incremental(
            proposal, changed, content_em, neural_lm, T_mcmc,
            glyph_word_idx, cur_lps)
        delta = prop_score - cur_score
        if delta > 0 or rng.random() < math.exp(delta * temp_scale):
            state, cur_score, cur_lps = proposal, prop_score, prop_lps
            n_accepted += 1
        if it % THIN == 0:
            chain_mappings.append(dict(state.mapping))
            chain_scores.append(cur_score)
        if it > 0 and it % 500 == 0:
            ar = n_accepted / (it + 1)
            if ar < 0.3: temp_scale *= 0.9
            elif ar > 0.7: temp_scale *= 1.1
    
    elapsed = time.time() - t0
    mcmc_chains.append({'mappings': chain_mappings, 'scores': chain_scores,
                        'accept_rate': n_accepted / N_ITER})
    print(f'  Chain {chain_id}: accept={n_accepted/N_ITER:.3f}, '
          f'final_score={cur_score:.4f}, {elapsed:.0f}s')

# Best from MCMC
global_best_map, global_best_score = None, -float('inf')
for ch in mcmc_chains:
    for m, s in zip(ch['mappings'], ch['scores']):
        if s > global_best_score:
            global_best_score, global_best_map = s, m

final_state = NucleusMappingState(CONTENT_VOCAB, target_chars)
final_state.set_mapping(global_best_map)
print(f'\nGlobal best MCMC score: {global_best_score:.4f}')
print(f'K_eff: {len(set(final_state.mapping.values()))}')

MCMC: 4 chains, 15000 iter, T=2.0
  Chain 0: accept=0.926, final_score=-25.9431, 15s
  Chain 1: accept=0.931, final_score=-25.7751, 14s
  Chain 2: accept=0.931, final_score=-25.7854, 14s
  Chain 3: accept=0.931, final_score=-25.9139, 14s

Global best MCMC score: -24.4242
K_eff: 27


In [7]:
# ── Cell 7: 5-test scorecard ─────────────────────────────────────────
from scipy.spatial.distance import jensenshannon

mapping_final = final_state.mapping

# ── R-hat ──
burn_idx = max(BURN_IN // THIN, 1)
rhat = {}
for g in CONTENT_VOCAB:
    all_post = [m.get(g) for ch in mcmc_chains
                for m in ch['mappings'][burn_idx:]]
    if not all_post: rhat[g] = float('nan'); continue
    top = Counter(all_post).most_common(1)[0][0]
    chain_means = []
    for ch in mcmc_chains:
        pb = ch['mappings'][burn_idx:]
        if pb:
            chain_means.append(np.mean([1.0 if m.get(g)==top else 0.0 for m in pb]))
    if len(chain_means) < 2: rhat[g] = float('nan'); continue
    gm = np.mean(chain_means)
    n_per = len(mcmc_chains[0]['mappings'][burn_idx:])
    B = n_per / (len(chain_means)-1) * sum((cm-gm)**2 for cm in chain_means)
    W_terms = []
    for ch in mcmc_chains:
        pb = ch['mappings'][burn_idx:]
        if pb:
            ind = [1.0 if m.get(g)==top else 0.0 for m in pb]
            W_terms.append(np.var(ind, ddof=1) if len(ind)>1 else 0.0)
    W = np.mean(W_terms)
    if W < 1e-12: rhat[g] = 1.0 if B < 1e-12 else float('inf')
    else: rhat[g] = math.sqrt(((1-1/n_per)*W + B/n_per) / W)

rhat_vals = [v for v in rhat.values() if not (np.isnan(v) or np.isinf(v))]
pct_converged = 100 * sum(1 for v in rhat_vals if v < 1.1) / max(len(rhat_vals), 1)
print(f'R-hat < 1.1: {pct_converged:.1f}% of {len(rhat_vals)} glyphs')

# ── Test 1: Collapse z-score ──
# Group content glyphs by target char
def glyph_word_pos_dist(g, content_words):
    counts = np.array([0.0, 0.0, 0.0])  # start, mid, end
    for w in content_words:
        for i, x in enumerate(w):
            if x == g:
                if len(w)==1: counts[0]+=0.5; counts[2]+=0.5
                elif i==0: counts[0]+=1
                elif i==len(w)-1: counts[2]+=1
                else: counts[1]+=1
    t = counts.sum()
    return counts/t if t>0 else np.array([1/3,1/3,1/3])

pos_dists = {g: glyph_word_pos_dist(g, words_content) for g in CONTENT_VOCAB}
groups = defaultdict(list)
for g, c in mapping_final.items():
    if g in CONTENT_VOCAB: groups[c].append(g)

z_scores = []
rng_null = np.random.default_rng(42)
for ch, gs in groups.items():
    if len(gs) < 3: continue
    pairs = list(combinations(gs, 2))
    real = np.mean([jensenshannon(pos_dists[a]+1e-10, pos_dists[b]+1e-10) for a,b in pairs])
    nulls = []
    for _ in range(200):
        rs = [CONTENT_VOCAB[rng_null.integers(len(CONTENT_VOCAB))] for _ in gs]
        rp = list(combinations(rs, 2))
        if rp: nulls.append(np.mean([jensenshannon(pos_dists[a]+1e-10, pos_dists[b]+1e-10) for a,b in rp]))
    if nulls:
        z = (real - np.mean(nulls)) / max(np.std(nulls), 1e-10)
        z_scores.append(z)

mean_z = np.mean(z_scores) if z_scores else 0.0
v1 = 'PASS' if mean_z > -1.0 else ('MARGINAL' if mean_z > -1.5 else 'FAIL')
print(f'\nTest 1 - Collapse z: {mean_z:.3f} → {v1}')

# ── Test 2: N-gram percentilee ──
decoded_corpus = final_state.decode_corpus(words_content)

def count_ngrams(words, n=2):
    c = Counter()
    for w in words:
        seq = ['<w>'] + list(w) + ['</w>']
        for i in range(len(seq)-n+1): c[tuple(seq[i:i+n])] += 1
    return c

ref_bi = set(count_ngrams(hebrew_words, 2).keys())
ref_tri = set(count_ngrams(hebrew_words, 3).keys())

def ngram_frac(decoded, ref_bi, ref_tri):
    tb, mb, tt, mt = 0, 0, 0, 0
    for w in decoded:
        seq = ['<w>'] + list(w) + ['</w>']
        for i in range(len(seq)-1):
            tb += 1
            if tuple(seq[i:i+2]) in ref_bi: mb += 1
        for i in range(len(seq)-2):
            tt += 1
            if tuple(seq[i:i+3]) in ref_tri: mt += 1
    return (mb/max(tb,1) + mt/max(tt,1)) / 2

real_ngram = ngram_frac(decoded_corpus, ref_bi, ref_tri)

null_ngrams = []
for _ in range(1000):
    null_map = {g: target_chars[rng_null.integers(len(target_chars))] for g in CONTENT_VOCAB}
    ns = NucleusMappingState(CONTENT_VOCAB, target_chars)
    ns.set_mapping(null_map)
    null_ngrams.append(ngram_frac(ns.decode_corpus(words_content), ref_bi, ref_tri))

ngram_pct = 100 * sum(1 for x in null_ngrams if real_ngram > x) / len(null_ngrams)
v2 = 'PASS' if ngram_pct > 15 else ('MARGINAL' if ngram_pct > 5 else 'FAIL')
print(f'Test 2 - N-gram: real={real_ngram:.4f}, pct={ngram_pct:.1f}% → {v2}')

# ── Test 3: Root matching ──
ROOTS = ['כתב','דבר','שמר','עשה','נתן','הלך','אמר','ידע','בוא','ראה',
         'שמע','לקח','עבד','מלך','ברך','קדש','זכר','שלח','ישב','עמד',
         'גדל','חיה','מות','אכל','שתה','לבש','רחץ','חכם','טהר','צדק',
         'שפט','משל','ספר','למד','בנה','פתח','סגר','קרא','נשא','שוב']

roots_found = set()
n_root_matches = 0
for w in decoded_corpus:
    ws = ''.join(w)
    for root in ROOTS:
        pos = 0; ok = True
        for ch in root:
            p = ws.find(ch, pos)
            if p == -1: ok = False; break
            pos = p + 1
        if ok: roots_found.add(root); n_root_matches += 1

v3 = 'PASS' if len(roots_found) >= 10 else ('MARGINAL' if len(roots_found) >= 5 else 'FAIL')
print(f'Test 3 - Roots: {len(roots_found)} found → {v3}')
if roots_found: print(f'  {sorted(roots_found)}')

# ── Test 4: Cardinality ──
K_eff = len(set(final_state.mapping.values()))
K_target = len(target_chars)
card_ratio = K_eff / K_target
v4 = 'PASS' if card_ratio > 0.6 else ('MARGINAL' if card_ratio > 0.4 else 'FAIL')
print(f'Test 4 - Cardinality: {K_eff}/{K_target} = {card_ratio:.3f} → {v4}')

# ── Test 5: Word length ──
dec_lens = [len(w) for w in decoded_corpus]
heb_lens = [len(w) for w in hebrew_words]
z_len = abs(np.mean(dec_lens) - np.mean(heb_lens)) / max(np.std(heb_lens), 1e-10)
v5 = 'PASS' if z_len < 2.0 else ('MARGINAL' if z_len < 3.0 else 'FAIL')
print(f'Test 5 - Word length: decoded={np.mean(dec_lens):.2f}, Hebrew={np.mean(heb_lens):.2f}, z={z_len:.2f} → {v5}')

# ── Summary ──
verdicts = [v1, v2, v3, v4, v5]
n_pass = sum(1 for v in verdicts if v=='PASS')
n_marg = sum(1 for v in verdicts if v=='MARGINAL')
n_fail = sum(1 for v in verdicts if v=='FAIL')
print(f'\n{"="*60}')
print(f'SCORECARD: {n_pass}+/{n_marg}±/{n_fail}−')
print(f'R-hat < 1.1: {pct_converged:.1f}%')
print(f'N-gram percentile: {ngram_pct:.1f}%')
print(f'Roots: {len(roots_found)}')
print(f'K_eff: {K_eff}/{K_target}')
print(f'{"="*60}')

R-hat < 1.1: 95.8% of 120 glyphs

Test 1 - Collapse z: -0.410 → PASS
Test 2 - N-gram: real=0.8640, pct=100.0% → PASS
Test 3 - Roots: 2 found → FAIL
  ['אכל', 'הלך']
Test 4 - Cardinality: 27/27 = 1.000 → PASS
Test 5 - Word length: decoded=2.22, Hebrew=3.81, z=1.31 → PASS

SCORECARD: 4+/0±/1−
R-hat < 1.1: 95.8%
N-gram percentile: 100.0%
Roots: 2
K_eff: 27/27


In [8]:
# ── Cell 8: Decoded samples + mapping analysis ────────────────────────

print('=== Sample decoded words ===')
print(f'{"Voynich word":25s} {"Nucleus only":15s} {"Decoded":15s}')
print('─' * 60)
for i in range(min(30, len(words))):
    v_str = ' '.join(words[i])
    n_str = ' '.join(words_content[i])
    d_str = ''.join(final_state.decode_word(words_content[i]))
    print(f'{v_str:25s} {n_str:15s} {d_str}')

print(f'\n=== Mapping ({len(mapping_final)} entries) ===')
v_freq = Counter(g for w in words_content for g in w)
for g in sorted(CONTENT_VOCAB, key=lambda g: v_freq.get(g,0), reverse=True)[:20]:
    hc = mapping_final.get(g, '?')
    info = glyph_info.get(g, {})
    print(f'  {g:4s} (freq={v_freq[g]:5d}, nuc={info.get("pct_nucleus",0):.0%}) → {hc}')

print(f'\n=== Position analysis ===')
print(f'Structural glyphs dropped: {len(STRUCTURAL_GLYPHS)}')
print(f'Content glyphs mapped: {len(CONTENT_VOCAB)}')
print(f'Hebrew chars used: {K_eff}/{K_target}')
print(f'Mean decoded word length: {np.mean(dec_lens):.2f} (Hebrew: {np.mean(heb_lens):.2f})')

=== Sample decoded words ===
Voynich word              Nucleus only    Decoded        
────────────────────────────────────────────────────────────
P2 A3 K1 A2 C2            C2              ה
A2 Q1 A3 B2               Q1 A3           אל
A3 C1                     A3 C1           לא
Aa Q2 A3 G1               Aa A3           מל
L1 A1 B2                  A1              ו
L1 A1 C1 A2               A1 C1           וא
U5 C2 J1 C2               J1 C2           לה
A2                        A2              ם
Q1 A1 C1                  Q1              א
L1 A1 B2 B1 A2            A1              ו
C2 A1 C1 A2               A1 C1           וא
U1 A3 C1                  U1              ה
A1 C1 A2                  C1              א
Q1 A3 F3                  F3              ם
K1 Q2 A3 G1               K1 A3           אל
L1 A3 C1                  L1              א
Aa E1 C2                  Aa              מ
U2 A3 C1                  U2              ה
U2 A3 C1                  U2              ה
B1 A3 E2 

In [9]:
# ── Cell 9: Export results ────────────────────────────────────────────

results = {
    'metadata': {
        'model': 'Nucleus-Only (Model 13)',
        'description': 'Only nucleus-position glyphs map to Hebrew chars; onset/coda are structural NULL',
        'content_vocab': len(CONTENT_VOCAB),
        'structural_glyphs': len(STRUCTURAL_GLYPHS),
        'onset_types': len(ONSET_GLYPHS),
        'nucleus_types': len(NUCLEUS_GLYPHS),
        'coda_types': len(CODA_GLYPHS),
        'ratio_sigma_A': round(len(CONTENT_VOCAB) / len(target_chars), 2),
        'mean_content_length': round(np.mean(content_lengths), 2),
        'mean_orig_length': round(np.mean(orig_lengths), 2),
        'lm': 'LSTM 2-layer (embed=32, hidden=192)',
        'temperature': T_mcmc,
        'n_seeds': N_SEEDS * len(T_VALUES),
    },
    'scorecard': {
        'collapse_z': round(mean_z, 3),
        'ngram_percentile': round(ngram_pct, 1),
        'roots_found': len(roots_found),
        'roots_list': sorted(roots_found),
        'cardinality_ratio': round(card_ratio, 3),
        'K_eff': K_eff,
        'word_length_z': round(z_len, 2),
        'rhat_pct': round(pct_converged, 1),
        'verdicts': verdicts,
        'summary': f'{n_pass}+/{n_marg}±/{n_fail}−',
    },
    'best_em': {
        'score': round(BEST['score'], 4),
        'temperature': BEST['T'],
        'seed': BEST['seed'],
        'K_eff': BEST['K_eff'],
    },
    'mcmc_best_score': round(global_best_score, 4),
    'val_loss': round(best_val_loss, 4),
}

with open('nucleus_resultados.json', 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print('Results saved to nucleus_results.json')
print(json.dumps(results, indent=2, ensure_ascii=False))

Results saved to nucleus_resultados.json
{
  "metadata": {
    "model": "Nucleus-Only (Model 13)",
    "description": "Only nucleus-position glyphs map to Hebrew chars; onset/coda are structural NULL",
    "content_vocab": 120,
    "structural_glyphs": 47,
    "onset_types": 42,
    "nucleus_types": 28,
    "coda_types": 5,
    "ratio_sigma_A": 4.44,
    "mean_content_length": 2.22,
    "mean_orig_length": 4.24,
    "lm": "LSTM 2-layer (embed=32, hidden=192)",
    "temperature": 2.0,
    "n_seeds": 20
  },
  "scorecard": {
    "collapse_z": -0.41,
    "ngram_percentile": 100.0,
    "roots_found": 2,
    "roots_list": [
      "אכל",
      "הלך"
    ],
    "cardinality_ratio": 1.0,
    "K_eff": 27,
    "word_length_z": 1.31,
    "rhat_pct": 95.8,
    "verdicts": [
      "PASS",
      "PASS",
      "FAIL",
      "PASS",
      "PASS"
    ],
    "summary": "4+/0±/1−"
  },
  "best_em": {
    "score": -24.3474,
    "temperature": 2.0,
    "seed": 0,
    "K_eff": 27
  },
  "mcmc_best_score": -